# Spot Check per Band Confidence — Validasi Kualitas Data Berlabel Otomatis

**Metode:** Confident Learning (Northcutt et al., 2019)  
**Ukuran sampel:** Formula Cochran (1977) — CI 95%, ME ±10%  
**Threshold kelayakan:** Akurasi ≥ 80% (Aribowo et al., 2025; Husaini et al., 2025)  
**Dasar noise tolerance:** Liu et al. (2022) — noise ≤ 20% dalam batas normal NLP  

## Alur Notebook
```
Cell 1  Setup
Cell 2  Load data & distribusi per band
Cell 3  Hitung ukuran sampel Cochran (1977)
Cell 4  Generate file CSV per band untuk cek manual
──────── HENTIKAN DI SINI — Isi CSV secara manual ────────
Cell 5  Hitung akurasi + Wilson CI per band
Cell 6  Keputusan kelayakan data
Cell 7  Visualisasi hasil
Cell 8  Tabel + kalimat otomatis untuk Bab 3 skripsi
```

## Definisi Band
| Band | Rentang Confidence | Interpretasi |
|---|---|---|
| Rendah | 0.00 – 0.70 | Model kurang yakin |
| Sedang | 0.70 – 0.85 | Model cukup yakin |
| Tinggi | 0.85 – 1.00 | Model sangat yakin |


## Cell 1 — Setup & Konfigurasi

In [14]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

# ── PATH ─────────────────────────────────────────────────────────────────────
BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
FILE_IN    = os.path.join(BASE_DIR, 'labelled_data_final.csv')
CHECK_DIR  = os.path.join(BASE_DIR, 'spot_check')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(CHECK_DIR,  exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# ── KONFIGURASI SPOT CHECK ────────────────────────────────────────────────────
# Definisi band confidence
# ── KONFIGURASI SAMPEL PER BAND (bisa berbeda-beda) ──────────────────────────
# Setiap band bisa punya margin of error berbeda
# Semakin kecil ME → semakin banyak sampel → semakin akurat
BAND_CONFIG = {
    'rendah': {
        'margin_err' : 0.07,   # ±7% → lebih ketat karena risiko tinggi
        'keterangan' : 'ME ±7%, CI 95% — lebih ketat karena confidence terendah'
    },
    'sedang': {
        'margin_err' : 0.10,   # ±10% → standar
        'keterangan' : 'ME ±10%, CI 95% — standar'
    },
    'tinggi': {
        'margin_err' : 0.10,   # ±10% → standar
        'keterangan' : 'ME ±10%, CI 95% — standar'
    },
}

# Parameter Cochran (1977)
Z_SCORE    = 1.96   # tingkat kepercayaan 95%
P_PROP     = 0.50   # worst-case proportion (variansi maksimum)
MARGIN_ERR = 0.10   # margin of error ±10%

# Threshold kelayakan
# Referensi: Aribowo et al. (2025); Husaini et al. (2025)
# Setara noise rate ≤ 20% — batas normal NLP (Liu et al., 2022)
THRESHOLD  = 80.0   # akurasi minimum (%)

SEED       = 42
CLASS_NAMES= ['keluhan', 'saran', 'pujian']

print('Setup selesai!')
print(f'Input       : {FILE_IN}')
print(f'Output CSV  : {CHECK_DIR}')
print(f'Threshold   : ≥ {THRESHOLD}% (Aribowo et al., 2025)')
print(f'Cochran     : Z={Z_SCORE}, ME=±{int(MARGIN_ERR*100)}%, CI=95%')

Setup selesai!
Input       : C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\labelled_data_final.csv
Output CSV  : C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\spot_check
Threshold   : ≥ 80.0% (Aribowo et al., 2025)
Cochran     : Z=1.96, ME=±10%, CI=95%


## Cell 2 — Load Data & Distribusi per Band

In [15]:
# ── Load data ────────────────────────────────────────────────────────────────
df_full   = pd.read_csv(FILE_IN)
df_full   = df_full.dropna(subset=['text','label_pks','confidence'])

df_manual = df_full[df_full['confidence'] == 1.0].copy()
df_auto   = df_full[df_full['confidence'] <  1.0].copy()

TOTAL_AUTO = len(df_auto)

print('='*60)
print('RINGKASAN DATA')
print('='*60)
print(f'Total data berlabel: {len(df_full):,}')
print(f'  Manual   (conf=1.0): {len(df_manual):,}  ← ground truth, tidak dispot check')
print(f'  Otomatis (conf<1.0): {TOTAL_AUTO:,}  ← yang akan divalidasi')

# ── Distribusi per band ───────────────────────────────────────────────────────
print(f'\n{"-"*60}')
print('DISTRIBUSI DATA OTOMATIS PER BAND CONFIDENCE')
print(f'{"-"*60}')

band_info = {}
for nama, bawah, atas, warna in BANDS:
    if nama == 'rendah':
        mask = df_auto['confidence'] <= atas
    else:
        mask = (df_auto['confidence'] > bawah) & (df_auto['confidence'] <= atas)

    df_band = df_auto[mask].copy()
    N       = len(df_band)
    pct     = N / TOTAL_AUTO * 100
    bar     = chr(9608) * int(pct / 3)

    lbl_dist = df_band['label_pks'].value_counts()

    band_info[nama] = {
        'bawah'   : bawah, 'atas': atas, 'warna': warna,
        'N'       : N, 'pct': pct,
        'df'      : df_band,
        'lbl_dist': lbl_dist,
    }

    print(f'\nBAND {nama.upper():7s} (conf {bawah:.2f}–{atas:.2f})')
    print(f'  Total data  : {N:7,} ({pct:.1f}%) {bar}')
    print(f'  Distribusi label:')
    for lbl in CLASS_NAMES:
        cnt  = lbl_dist.get(lbl, 0)
        lpct = cnt / N * 100 if N > 0 else 0
        print(f'    {lbl:10s}: {cnt:6,} ({lpct:.1f}%)')
    conf_band = df_band['confidence']
    print(f'  Confidence  : min={conf_band.min():.4f}  '
          f'max={conf_band.max():.4f}  mean={conf_band.mean():.4f}')

print(f'\nTotal otomatis: {TOTAL_AUTO:,}')

RINGKASAN DATA
Total data berlabel: 80,163
  Manual   (conf=1.0): 18,534  ← ground truth, tidak dispot check
  Otomatis (conf<1.0): 61,629  ← yang akan divalidasi

------------------------------------------------------------
DISTRIBUSI DATA OTOMATIS PER BAND CONFIDENCE
------------------------------------------------------------

BAND RENDAH  (conf 0.00–0.70)
  Total data  :   8,511 (13.8%) ████
  Distribusi label:
    keluhan   :  3,746 (44.0%)
    saran     :  3,075 (36.1%)
    pujian    :  1,690 (19.9%)
  Confidence  : min=0.3359  max=0.7000  mean=0.5727

BAND SEDANG  (conf 0.70–0.85)
  Total data  :   7,303 (11.8%) ███
  Distribusi label:
    keluhan   :  3,716 (50.9%)
    saran     :  2,549 (34.9%)
    pujian    :  1,038 (14.2%)
  Confidence  : min=0.7001  max=0.8500  mean=0.7835

BAND TINGGI  (conf 0.85–1.00)
  Total data  :  45,815 (74.3%) ████████████████████████
  Distribusi label:
    keluhan   : 30,663 (66.9%)
    saran     :  5,855 (12.8%)
    pujian    :  9,297 (20.3%)
  C

## Cell 3 — Hitung Ukuran Sampel dengan Formula Cochran (1977)

$$n = \frac{Z^2 \times p \times (1-p)}{E^2}$$

Koreksi populasi terbatas (*Finite Population Correction*):
$$n_{adj} = \frac{n}{1 + \dfrac{n-1}{N}}$$

**Referensi:** Cochran, W. G. (1977). *Sampling Techniques* (3rd ed.). Wiley.


In [16]:
print('='*60)
print('UKURAN SAMPEL COCHRAN (1977) — BERBEDA PER BAND')
print(f'Parameter: Z={Z_SCORE} (CI 95%), p={P_PROP}')
print('='*60)

total_sampel = 0
for nama, bawah, atas, _ in BANDS:
    N   = band_info[nama]['N']
    ME  = BAND_CONFIG[nama]['margin_err']
    ket = BAND_CONFIG[nama]['keterangan']

    # Formula Cochran (1977)
    n_base = (Z_SCORE**2 * P_PROP * (1-P_PROP)) / (ME**2)
    n_adj  = n_base / (1 + (n_base-1)/N)
    n      = int(np.ceil(n_adj))
    n      = min(n, N)

    pct_n = n / N * 100
    waktu = n * 2.5 / 60

    band_info[nama]['n_sampel']  = n
    band_info[nama]['margin_err']= ME
    total_sampel += n

    print(f'\nBand {nama.upper():7s} (N={N:,})')
    print(f'  Justifikasi : {ket}')
    print(f'  n sampel    : {n} ({pct_n:.2f}% dari band)')
    print(f'  Estimasi    : ~{waktu:.1f} jam spot check manual')

waktu_total = total_sampel * 2.5 / 60
print(f'\n{"─"*40}')
print(f'Total sampel   : {total_sampel} baris')
print(f'Total waktu    : ~{waktu_total:.1f} jam (@2.5 menit/baris)')
print(f'\nKonfigurasi:')
print(f'  Band rendah  → ME ±{int(BAND_CONFIG["rendah"]["margin_err"]*100)}%  '
      f'(lebih ketat, risiko tinggi)')
print(f'  Band sedang  → ME ±{int(BAND_CONFIG["sedang"]["margin_err"]*100)}%  '
      f'(standar)')
print(f'  Band tinggi  → ME ±{int(BAND_CONFIG["tinggi"]["margin_err"]*100)}%  '
      f'(standar)')

UKURAN SAMPEL COCHRAN (1977) — BERBEDA PER BAND
Parameter: Z=1.96 (CI 95%), p=0.5

Band RENDAH  (N=8,511)
  Justifikasi : ME ±7%, CI 95% — lebih ketat karena confidence terendah
  n sampel    : 192 (2.26% dari band)
  Estimasi    : ~8.0 jam spot check manual

Band SEDANG  (N=7,303)
  Justifikasi : ME ±10%, CI 95% — standar
  n sampel    : 95 (1.30% dari band)
  Estimasi    : ~4.0 jam spot check manual

Band TINGGI  (N=45,815)
  Justifikasi : ME ±10%, CI 95% — standar
  n sampel    : 96 (0.21% dari band)
  Estimasi    : ~4.0 jam spot check manual

────────────────────────────────────────
Total sampel   : 383 baris
Total waktu    : ~16.0 jam (@2.5 menit/baris)

Konfigurasi:
  Band rendah  → ME ±7%  (lebih ketat, risiko tinggi)
  Band sedang  → ME ±10%  (standar)
  Band tinggi  → ME ±10%  (standar)


## Cell 4 — Generate File CSV per Band untuk Spot Check Manual
**Stratified sampling per label** — proporsi sampel tiap kelas sesuai distribusi band.
> Output: 3 file CSV di folder `spot_check/`  
> Kolom `label_spot_check` dikosongkan — **isi oleh peneliti saat cek manual**


In [17]:
print('Membuat file spot check...')
print('='*60)

for nama, bawah, atas, _ in BANDS:
    df_band  = band_info[nama]['df']
    n_target = band_info[nama]['n_sampel']
    N        = band_info[nama]['N']

    # Stratified sampling per label agar tiap kelas terwakili proporsional
    samples = []
    for lbl in CLASS_NAMES:
        df_lbl = df_band[df_band['label_pks'] == lbl]
        if len(df_lbl) == 0:
            continue
        # proporsi sampel per kelas = proporsi kelas di band
        n_lbl = max(1, round(n_target * len(df_lbl) / N))
        n_lbl = min(n_lbl, len(df_lbl))
        samples.append(df_lbl.sample(n_lbl, random_state=SEED))

    df_samp = pd.concat(samples, ignore_index=True)

    # Sesuaikan ke n_target
    if len(df_samp) > n_target:
        df_samp = df_samp.sample(n_target, random_state=SEED)
    elif len(df_samp) < n_target and len(df_band) >= n_target:
        tambahan = df_band[~df_band.index.isin(df_samp.index)]
        df_samp  = pd.concat([
            df_samp,
            tambahan.sample(n_target - len(df_samp), random_state=SEED)
        ], ignore_index=True)

    # Acak urutan baris
    df_samp = df_samp.sample(frac=1, random_state=SEED).reset_index(drop=True)
    df_samp.insert(0, 'no', range(1, len(df_samp)+1))

    # Kolom output
    # label_spot_check = KOSONG → diisi peneliti saat cek manual
    # keterangan       = KOSONG → opsional, catatan peneliti
    df_samp['label_spot_check'] = ''  # ← ISI: keluhan / saran / pujian
    df_samp['keterangan']       = ''  # ← opsional

    cols_out = ['no', 'text', 'label_pks', 'confidence',
                'label_spot_check', 'keterangan']
    df_out = df_samp[cols_out].copy()

    fname = os.path.join(CHECK_DIR, f'spot_check_band_{nama}.csv')
    df_out.to_csv(fname, index=False, encoding='utf-8-sig')

    print(f'Band {nama.upper():7s}: {len(df_out):3d} sampel tersimpan')
    print(f'  File: {fname}')
    print(f'  Distribusi sampel:')
    for lbl, cnt in df_out['label_pks'].value_counts().items():
        print(f'    {lbl:10s}: {cnt}')
    print()

print('='*60)
print('SEMUA FILE TERSIMPAN!')
print('='*60)
print()
print('LANGKAH SELANJUTNYA — CARA MENGISI SPOT CHECK:')
print('  1. Buka setiap file CSV di Microsoft Excel')
print('  2. Baca kolom TEXT dengan seksama')
print('  3. Isi kolom LABEL_SPOT_CHECK dengan salah satu:')
print('       keluhan  →  ekspresi ketidakpuasan, masalah, komplain')
print('       saran    →  usulan, rekomendasi, harapan perbaikan')
print('       pujian   →  apresiasi, terima kasih, puas')
print('  4. Kolom KETERANGAN opsional (isi jika ada catatan)')
print('  5. Simpan file dengan nama yang SAMA (jangan rename)')
print('  6. Jalankan Cell 5 untuk menghitung akurasi')
print()
print(f'Total yang perlu dicek: {sum(b["n_sampel"] for b in band_info.values())} baris')

Membuat file spot check...
Band RENDAH : 192 sampel tersimpan
  File: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\spot_check\spot_check_band_rendah.csv
  Distribusi sampel:
    keluhan   : 85
    saran     : 69
    pujian    : 38

Band SEDANG :  95 sampel tersimpan
  File: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\spot_check\spot_check_band_sedang.csv
  Distribusi sampel:
    keluhan   : 48
    saran     : 33
    pujian    : 14

Band TINGGI :  96 sampel tersimpan
  File: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\spot_check\spot_check_band_tinggi.csv
  Distribusi sampel:
    keluhan   : 64
    pujian    : 20
    saran     : 12

SEMUA FILE TERSIMPAN!

LANGKAH SELANJUTNYA — CARA MENGISI SPOT CHECK:
  1. Buka setiap file CSV di Microsoft Excel
  2. Baca kolom TEXT dengan seksama
  3. Isi kolom LABEL_SPOT_CHECK dengan salah satu:
       keluhan  →  ekspresi ketidakpuasan, masalah, komplain
       saran    →  usulan, rekomendasi, hara

## Cell 5 — Hitung Akurasi + Wilson Confidence Interval per Band
> **Jalankan SETELAH semua file CSV diisi manual dan disimpan**

**Wilson CI** digunakan karena lebih akurat untuk proporsi
dibanding Wald CI standar, terutama saat n kecil (Brown et al., 2001).


In [23]:
from statsmodels.stats.proportion import proportion_confint


print('='*60)
print('HASIL SPOT CHECK PER BAND')
print('='*60)

hasil     = []
ada_error = False

# Label yang dianggap tidak bisa dinilai (noise/teks tidak bermakna)
LABEL_NOISE = {'tidak_valid', 'noise', 'ambigu', 'tidak dapat dinilai'}

for nama, bawah, atas, warna in BANDS:
    fname = os.path.join(CHECK_DIR, f'spot_check_band_{nama}.csv')

    # ── Cek file ada ──────────────────────────────────────────────
    if not os.path.exists(fname):
        print(f'Band {nama}: FILE TIDAK DITEMUKAN — {fname}')
        ada_error = True
        continue

    df_cek = pd.read_csv(fname)

    # ── Filter baris yang sudah diisi ────────────────────────────
    df_filled = df_cek[
        df_cek['label_spot_check'].notna() &
        (df_cek['label_spot_check'].astype(str).str.strip() != '')
    ].copy()

    if len(df_filled) == 0:
        print(f'Band {nama}: BELUM DIISI — isi label_spot_check dulu')
        ada_error = True
        continue

    # ── Normalisasi label ─────────────────────────────────────────
    df_filled['label_pks']        = df_filled['label_pks'].str.lower().str.strip()
    df_filled['label_spot_check'] = df_filled['label_spot_check'].str.lower().str.strip()

    # ── Pisahkan: noise vs dapat dinilai ─────────────────────────
    df_noise  = df_filled[df_filled['label_spot_check'].isin(LABEL_NOISE)].copy()
    df_valid  = df_filled[~df_filled['label_spot_check'].isin(LABEL_NOISE)].copy()

    # ── Validasi label yang dapat dinilai ────────────────────────
    label_aneh = df_valid[
        ~df_valid['label_spot_check'].isin(set(CLASS_NAMES))
    ]['label_spot_check'].unique()

    if len(label_aneh) > 0:
        print(f'Band {nama}: Label tidak dikenal: {label_aneh}')
        print(f'  Pastikan hanya isi: keluhan / saran / pujian / tidak_valid')
        ada_error = True
        continue

    # ── Hitung akurasi (HANYA dari data yang bisa dinilai) ───────
    n       = len(df_valid)   # excludes noise
    n_noise = len(df_noise)
    n_total = len(df_filled)

    if n == 0:
        print(f'Band {nama}: Semua sampel tidak dapat dinilai (tidak_valid)')
        ada_error = True
        continue

    benar      = (df_valid['label_pks'] == df_valid['label_spot_check']).sum()
    salah      = n - benar
    acc        = benar / n * 100
    noise_rate = salah / n * 100

    # ── Wilson Confidence Interval (95%) ─────────────────────────
    # Referensi: Brown et al. (2001)
    ci_lo, ci_hi = proportion_confint(
        count=benar, nobs=n, alpha=0.05, method='wilson'
    )
    ci_lo *= 100; ci_hi *= 100

    # ── Status kelayakan ─────────────────────────────────────────
    layak  = acc >= THRESHOLD
    status = '✅ LAYAK' if layak else '❌ EXCLUDE'

    # ── Tampilkan hasil ──────────────────────────────────────────
    print(f'\nBAND {nama.upper()} (confidence {bawah:.2f}–{atas:.2f})')
    print(f'  N populasi      : {band_info[nama]["N"]:,}')
    print(f'  N total dicek   : {n_total}')
    if n_noise > 0:
        print(f'  N tidak_valid   : {n_noise} '
              f'({n_noise/n_total*100:.1f}%) ← teks tidak bermakna, diexclude dari hitung akurasi')
    print(f'  N dapat dinilai : {n}')
    print(f'  Label benar     : {benar} ({acc:.2f}%)')
    print(f'  Label salah     : {salah} ({noise_rate:.2f}%) ← noise rate')
    print(f'  Akurasi         : {acc:.2f}%')
    print(f'  95% Wilson CI   : [{ci_lo:.1f}% – {ci_hi:.1f}%]')
    print(f'  Status          : {status}  (threshold ≥{THRESHOLD}%)')

    # ── Detail akurasi per kelas ─────────────────────────────────
    print(f'  Akurasi per kelas:')
    for lbl in CLASS_NAMES:
        df_lbl = df_valid[df_valid['label_pks'] == lbl]
        if len(df_lbl) == 0:
            continue
        acc_lbl = (df_lbl['label_pks'] == df_lbl['label_spot_check']).sum() / len(df_lbl) * 100
        print(f'    {lbl:10s}: {len(df_lbl):3d} sampel → {acc_lbl:.1f}%')

    # ── Contoh label salah ───────────────────────────────────────
    df_salah = df_valid[df_valid['label_pks'] != df_valid['label_spot_check']]
    if len(df_salah) > 0:
        print(f'  Contoh label salah (maks 3):')
        for _, row in df_salah.head(3).iterrows():
            teks = str(row['text'])[:60]
            print(f'    Model:{row["label_pks"]:10s} → Manual:{row["label_spot_check"]}')
            print(f'    Teks : "{teks}"')

    # ── Contoh data tidak_valid ──────────────────────────────────
    if n_noise > 0:
        print(f'  Contoh data tidak_valid (maks 3):')
        for _, row in df_noise.head(3).iterrows():
            print(f'    "{str(row["text"])[:50]}" → {row["label_pks"]}')

    hasil.append({
        'band'       : nama,
        'bawah'      : bawah,
        'atas'       : atas,
        'N_pop'      : band_info[nama]['N'],
        'n_total'    : n_total,
        'n_noise'    : n_noise,
        'n_valid'    : n,
        'benar'      : benar,
        'salah'      : salah,
        'akurasi'    : acc,
        'noise_rate' : noise_rate,
        'ci_lo'      : ci_lo,
        'ci_hi'      : ci_hi,
        'layak'      : layak,
        'status'     : 'Layak' if layak else 'Exclude',
    })

if ada_error:
    print('\n⚠️  Ada band yang belum lengkap. Lengkapi dulu sebelum lanjut.')
else:
    total_noise = sum(h['n_noise'] for h in hasil)
    print(f'\n{"─"*60}')
    print(f'Semua band selesai dicek!')
    if total_noise > 0:
        print(f'Total data tidak_valid ditemukan: {total_noise} baris')
        print(f'→ Data ini akan diexclude dari dataset training di Cell 6')
    print(f'Lanjutkan ke Cell 6.')

HASIL SPOT CHECK PER BAND

BAND RENDAH (confidence 0.00–0.70)
  N populasi      : 8,511
  N total dicek   : 192
  N tidak_valid   : 20 (10.4%) ← teks tidak bermakna, diexclude dari hitung akurasi
  N dapat dinilai : 172
  Label benar     : 129 (75.00%)
  Label salah     : 43 (25.00%) ← noise rate
  Akurasi         : 75.00%
  95% Wilson CI   : [68.0% – 80.9%]
  Status          : ❌ EXCLUDE  (threshold ≥80.0%)
  Akurasi per kelas:
    keluhan   :  79 sampel → 83.5%
    saran     :  62 sampel → 77.4%
    pujian    :  31 sampel → 48.4%
  Contoh label salah (maks 3):
    Model:saran      → Manual:keluhan
    Teks : "emang awalnya berapa harii ya? soalnya saya cmn menerima mbg"
    Model:pujian     → Manual:keluhan
    Teks : "makan tu mbg mbg"
    Model:keluhan    → Manual:saran
    Teks : "Mending kasih uang aja perbulan, MBG itu banyak memakai uang"
  Contoh data tidak_valid (maks 3):
    "Udah . nih" → keluhan
    "singkong" → keluhan
    "NAGRIKALER" → keluhan

BAND SEDANG (confidence 0.

## Cell 6 — Keputusan Kelayakan Data & Buat Dataset Final
**Keputusan berdasarkan:**
- Threshold ≥ 80% → Layak (Aribowo et al., 2025; Husaini et al., 2025)
- Noise ≤ 20% → Dalam batas normal NLP (Liu et al., 2022)
- Model robust terhadap noise ≤ 20% (Ahmed et al., 2024; Agro et al., 2023)


In [24]:
if not hasil:
    print('Jalankan Cell 5 dulu!')
else:
    print('='*60)
    print('KEPUTUSAN KELAYAKAN DATA PER BAND')
    print('='*60)

    bands_layak   = [h for h in hasil if h['layak']]
    bands_exclude = [h for h in hasil if not h['layak']]

    for h in hasil:
        ikon = '✅' if h['layak'] else '❌'
        print(f'  {ikon} Band {h["band"]:7s}: '
              f'{h["akurasi"]:.2f}%  '
              f'[CI: {h["ci_lo"]:.1f}%–{h["ci_hi"]:.1f}%]  '
              f'Noise={h["noise_rate"]:.2f}%  '
              f'→ {h["status"]}')

    print(f'\n{"="*60}')
    print(f'KESIMPULAN')
    print(f'{"="*60}')

    if not bands_exclude:
        print('\n✅ SEMUA BAND MEMENUHI THRESHOLD ≥80%')
        print('   Seluruh 61.629 data berlabel otomatis LAYAK digunakan.')
        print('   Tidak ada data yang perlu diexclude.')
        print(f'   Noise rate tertinggi: '
              f'{max(h["noise_rate"] for h in hasil):.2f}% '
              f'< 20% (batas normal NLP, Liu et al., 2022)')
        dataset_final = 'labelled_data_final.csv'  # pakai dataset asli
        print(f'\n   Dataset yang digunakan: {dataset_final} (tidak perlu modifikasi)')

    else:
        print(f'\n⚠️  ADA BAND YANG TIDAK MEMENUHI THRESHOLD')
        for h in bands_exclude:
            print(f'   ❌ Band {h["band"]}: {h["akurasi"]:.2f}% < {THRESHOLD}% '
                  f'→ {band_info[h["band"]]["N"]:,} data diexclude')

        print(f'\n   Membuat dataset filtered...')

        # Exclude band yang tidak layak
        df_auto_valid = df_auto.copy()
        for h in bands_exclude:
            bawah, atas = h['bawah'], h['atas']
            if h['band'] == 'rendah':
                df_auto_valid = df_auto_valid[df_auto_valid['confidence'] > atas]
            else:
                df_auto_valid = df_auto_valid[
                    ~((df_auto_valid['confidence'] > bawah) &
                      (df_auto_valid['confidence'] <= atas))
                ]

        df_filtered = pd.concat([df_manual, df_auto_valid], ignore_index=True)
        path_filt   = os.path.join(BASE_DIR, 'labelled_data_filtered.csv')
        df_filtered.to_csv(path_filt, index=False, encoding='utf-8-sig')

        print(f'\n   Dataset asli    : {len(df_full):,} baris')
        print(f'   Dataset filtered: {len(df_filtered):,} baris '
              f'(berkurang {len(df_full)-len(df_filtered):,})')
        print(f'   Tersimpan       : {path_filt}')
        print(f'\n   Gunakan labelled_data_filtered.csv untuk modelling.')

KEPUTUSAN KELAYAKAN DATA PER BAND
  ❌ Band rendah : 75.00%  [CI: 68.0%–80.9%]  Noise=25.00%  → Exclude
  ❌ Band sedang : 78.41%  [CI: 68.7%–85.7%]  Noise=21.59%  → Exclude
  ✅ Band tinggi : 93.62%  [CI: 86.8%–97.0%]  Noise=6.38%  → Layak

KESIMPULAN

⚠️  ADA BAND YANG TIDAK MEMENUHI THRESHOLD
   ❌ Band rendah: 75.00% < 80.0% → 8,511 data diexclude
   ❌ Band sedang: 78.41% < 80.0% → 7,303 data diexclude

   Membuat dataset filtered...

   Dataset asli    : 80,163 baris
   Dataset filtered: 64,349 baris (berkurang 15,814)
   Tersimpan       : C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\labelled_data_filtered.csv

   Gunakan labelled_data_filtered.csv untuk modelling.


## Cell 7 — Visualisasi Hasil Spot Check

In [27]:
if not hasil:
    print('Jalankan Cell 5 dulu!')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        'Validasi Kualitas Label Otomatis — Spot Check per Band Confidence\n'
        '(Confident Learning, Northcutt et al., 2019)',
        fontweight='bold', fontsize=11
    )

    band_names  = [h['band'].capitalize() for h in hasil]
    akurasi_val = [h['akurasi'] for h in hasil]
    noise_val   = [h['noise_rate'] for h in hasil]
    ci_lo_val   = [h['akurasi'] - h['ci_lo'] for h in hasil]
    ci_hi_val   = [h['ci_hi'] - h['akurasi'] for h in hasil]
    colors_bar  = ['#27AE60' if h['layak'] else '#E74C3C' for h in hasil]

    # ── Plot 1: Akurasi per band + CI ─────────────────────────────
    ax = axes[0]
    x  = np.arange(len(hasil))
    bars = ax.bar(x, akurasi_val, color=colors_bar,
                  edgecolor='white', width=0.5, zorder=3)
    ax.errorbar(x, akurasi_val,
                yerr=[ci_lo_val, ci_hi_val],
                fmt='none', color='#2C3E50',
                capsize=6, capthick=2, lw=2, zorder=4)
    ax.axhline(THRESHOLD, color='#E74C3C', ls='--', lw=1.5,
               label=f'Threshold {THRESHOLD}%\n(Aribowo et al., 2025)')
    for bar, val in zip(bars, akurasi_val):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+2,
                f'{val:.1f}%',
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(band_names)
    ax.set_title('Akurasi per Band (95% Wilson CI)', fontweight='bold')
    ax.set_ylabel('Akurasi (%)'); ax.set_ylim(0, 115)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── Plot 2: Noise rate per band ────────────────────────────────
    ax = axes[1]
    colors_noise = ['#E74C3C' if n>20 else '#F39C12' if n>10 else '#27AE60'
                    for n in noise_val]
    bars2 = ax.bar(x, noise_val, color=colors_noise,
                   edgecolor='white', width=0.5)
    ax.axhline(20, color='#E74C3C', ls='--', lw=1.5,
               label='Batas normal 20%\n(Liu et al., 2022)')
    for bar, val in zip(bars2, noise_val):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.3,
                f'{val:.1f}%',
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(band_names)
    ax.set_title('Noise Rate per Band\n(% label salah)', fontweight='bold')
    ax.set_ylabel('Noise Rate (%)'); ax.set_ylim(0, 45)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── Plot 3: Proporsi benar vs salah ───────────────────────────
    ax = axes[2]
    # Gunakan n_valid (baris yang bisa dinilai, exclude tidak_valid)
    pct_benar = [h['benar']/h['n_valid']*100 for h in hasil]
    pct_salah = [h['salah']/h['n_valid']*100 for h in hasil]
    ax.bar(x, pct_benar, color='#27AE60', label='Label Benar',
           edgecolor='white', width=0.5)
    ax.bar(x, pct_salah, bottom=pct_benar, color='#E74C3C',
           label='Label Salah', edgecolor='white', width=0.5)
    for i, (b, s) in enumerate(zip(pct_benar, pct_salah)):
        if b > 5:
            ax.text(x[i], b/2, f'{b:.0f}%',
                    ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white')
        if s > 5:
            ax.text(x[i], b+s/2, f'{s:.0f}%',
                    ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white')
    ax.set_xticks(x); ax.set_xticklabels(band_names)
    ax.set_title('Proporsi Label\nBenar vs Salah', fontweight='bold')
    ax.set_ylabel('%'); ax.set_ylim(0, 115)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    path_plot = os.path.join(RESULT_DIR, 'spot_check_hasil.png')
    plt.savefig(path_plot, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Plot tersimpan: {path_plot}')

Plot tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\spot_check_hasil.png


## Cell 8 — Tabel Ringkasan + Kalimat Otomatis untuk Bab 3 Skripsi
> Salin tabel dan kalimat langsung ke dokumen skripsi


In [29]:
if not hasil:
    print('Jalankan Cell 5 dulu!')
else:
    # ── Tabel ringkasan ───────────────────────────────────────────
    rows = []
    for h in hasil:
        rows.append({
            'Band'               : h['band'].capitalize(),
            'Rentang Confidence' : f"{h['bawah']:.2f}–{h['atas']:.2f}",
            'N Data'             : f"{h['N_pop']:,}",
            'N Total Dicek'      : h['n_total'],
            'N Tidak Valid'      : h['n_noise'],
            'N Dapat Dinilai'    : h['n_valid'],
            'Label Benar'        : h['benar'],
            'Label Salah'        : h['salah'],
            'Akurasi (%)'        : f"{h['akurasi']:.2f}%",
            '95% Wilson CI'      : f"[{h['ci_lo']:.1f}%–{h['ci_hi']:.1f}%]",
            'Noise Rate (%)'     : f"{h['noise_rate']:.2f}%",
            'Kelayakan'          : h['status'],
        })

    df_tabel = pd.DataFrame(rows)
    path_tabel = os.path.join(RESULT_DIR, 'tabel_spot_check.csv')
    df_tabel.to_csv(path_tabel, index=False, encoding='utf-8-sig')

    print('TABEL HASIL SPOT CHECK PER BAND CONFIDENCE')
    print('(Salin ke Bab 3 Skripsi)')
    print('='*90)
    print(df_tabel.to_string(index=False))
    print(f'\nTersimpan: {path_tabel}')

    # ── Kalimat otomatis untuk skripsi ────────────────────────────
    all_layak    = all(h['layak'] for h in hasil)
    avg_acc      = np.mean([h['akurasi'] for h in hasil])
    max_noise    = max(h['noise_rate'] for h in hasil)
    total_samp   = sum(h['n_total'] for h in hasil)
    total_noise  = sum(h['n_noise'] for h in hasil)
    total_valid  = sum(h['n_valid'] for h in hasil)
    bands_ex     = [h for h in hasil if not h['layak']]
    bands_ok     = [h for h in hasil if h['layak']]

    print(f'\n{"="*60}')
    print('KALIMAT UNTUK BAB 3 SKRIPSI (COPY-PASTE LANGSUNG):')
    print(f'{"="*60}')

    # Bagian 1 — Deskripsi metodologi
    kalimat = f"""
Validasi kualitas data berlabel otomatis dilakukan melalui
spot check per band confidence mengacu pada kerangka Confident
Learning (Northcutt et al., 2019). Data berlabel otomatis
sebanyak {sum(b['N_pop'] for b in hasil):,} baris dibagi ke dalam
tiga band berdasarkan nilai confidence score, yaitu band rendah
(confidence ≤0,70), band sedang (0,70–0,85), dan band tinggi
(0,85–1,00).

Ukuran sampel pada masing-masing band ditentukan menggunakan
formula Cochran (1977) dengan tingkat kepercayaan 95% dan margin
of error berbeda per band sesuai tingkat risiko, menghasilkan
total {total_samp} sampel yang diperiksa secara manual oleh
peneliti. Dari total sampel tersebut, ditemukan {total_noise}
baris ({total_noise/total_samp*100:.1f}%) berupa teks tidak
bermakna (noise data) yang tidak dapat dinilai sentimen-nya,
sehingga evaluasi dilakukan terhadap {total_valid} sampel yang
dapat dinilai.
"""

    # Bagian 2 — Hasil per band
    kalimat += "\nHasil verifikasi manual menunjukkan bahwa:\n"
    for h in hasil:
        kalimat += (
            f"  - Band {h['band']} (confidence {h['bawah']:.2f}–{h['atas']:.2f}): "
            f"akurasi {h['akurasi']:.2f}% "
            f"(95% Wilson CI: [{h['ci_lo']:.1f}%–{h['ci_hi']:.1f}%]), "
            f"noise rate {h['noise_rate']:.2f}%, "
            f"status {h['status']}.\n"
        )

    # Bagian 3 — Keputusan
    if all_layak:
        kalimat += f"""
Seluruh band confidence memenuhi threshold kelayakan ≥80%
(Aribowo et al., 2025; Husaini et al., 2025), sehingga
keseluruhan data berlabel otomatis dinyatakan layak digunakan
sebagai data latih. Noise rate tertinggi sebesar {max_noise:.2f}%
berada dalam rentang noise yang wajar pada dataset NLP
(Liu et al., 2022), sehingga tidak mengganggu kemampuan model
untuk belajar secara efektif (Ahmed et al., 2024).
"""
    else:
        n_excluded = sum(b['N_pop'] for b in bands_ex)
        n_retained = sum(b['N_pop'] for b in bands_ok)
        nama_ex    = ', '.join([b['band'] for b in bands_ex])
        nama_ok    = ', '.join([b['band'] for b in bands_ok])

        kalimat += f"""
Band {nama_ex} tidak memenuhi threshold kelayakan ≥80%
(Aribowo et al., 2025; Husaini et al., 2025), sehingga
{n_excluded:,} data pada band tersebut diexclude dari dataset
training untuk mengurangi noise label (Northcutt et al., 2019).
Data pada band {nama_ok} dengan akurasi di atas threshold
dipertahankan ({n_retained:,} data), karena noise rate-nya
masih dalam batas toleransi yang dapat ditangani model
(Liu et al., 2022; Ahmed et al., 2024).

Dataset akhir yang digunakan untuk pelatihan model terdiri dari
{n_retained + len(df_manual):,} baris, yaitu {len(df_manual):,}
data berlabel manual dan {n_retained:,} data berlabel otomatis
dari band confidence yang dinyatakan layak.
"""

    print(kalimat)

    # Simpan kalimat ke file txt
    path_txt = os.path.join(RESULT_DIR, 'kalimat_spot_check_bab3.txt')
    with open(path_txt, 'w', encoding='utf-8') as f:
        f.write(kalimat)
    print(f'\nKalimat tersimpan: {path_txt}')
    print(f'Tabel tersimpan  : {path_tabel}')

TABEL HASIL SPOT CHECK PER BAND CONFIDENCE
(Salin ke Bab 3 Skripsi)
  Band Rentang Confidence N Data  N Total Dicek  N Tidak Valid  N Dapat Dinilai  Label Benar  Label Salah Akurasi (%) 95% Wilson CI Noise Rate (%) Kelayakan
Rendah          0.00–0.70  8,511            192             20              172          129           43      75.00% [68.0%–80.9%]         25.00%   Exclude
Sedang          0.70–0.85  7,303             95              7               88           69           19      78.41% [68.7%–85.7%]         21.59%   Exclude
Tinggi          0.85–1.00 45,815             96              2               94           88            6      93.62% [86.8%–97.0%]          6.38%     Layak

Tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\tabel_spot_check.csv

KALIMAT UNTUK BAB 3 SKRIPSI (COPY-PASTE LANGSUNG):

Validasi kualitas data berlabel otomatis dilakukan melalui
spot check per band confidence mengacu pada kerangka Confident
Learning (Northcutt et al.